In [1]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - Machine Measurements

In [4]:
# Load the data
machine_measurements_query = """
SELECT  mm.subject_id,
        mm.report_0,
        mm.report_1,
        mm.report_2,
        mm.report_3,
        mm.report_4,
        mm.report_5,
        mm.report_6,
        mm.report_7,
        mm.report_8,
        mm.report_9,
        mm.report_10,
        mm.report_11,
        mm.report_12,
        mm.report_13,
        mm.report_14,
        mm.report_15,
        mm.report_16,
        mm.report_17

FROM    physionet-data.mimiciv_ecg.machine_measurements mm
"""

machine_measurements_df = client.query(machine_measurements_query).to_dataframe()

print(machine_measurements_df.shape)
print(machine_measurements_df.dtypes)

(800035, 19)
subject_id     Int64
report_0      object
report_1      object
report_2      object
report_3      object
report_4      object
report_5      object
report_6      object
report_7      object
report_8      object
report_9      object
report_10     object
report_11     object
report_12     object
report_13     object
report_14     object
report_15     object
report_16     object
report_17     object
dtype: object


In [5]:
# Define regex patterns
# Each report_i column is an independent ECG machine finding.
# We scan each column separately to preserve finding count and avoid
# cross-column boundary artifacts from concatenation.

negative_patterns = re.compile(r"""
    ACUTE\s+INFARCT|
    INFARCT(?:ION)?|
    ISCH[EAE]MIA|
    MYOCARDIAL\s+INJURY|
    STRONGLY\s+SUGGESTS|
    CONSIDER\s+ACUTE|
    POSSIBLY\s+ACUTE|
    PROBABLY\s+ACUTE|
    COMPLETE\s+AV\s+BLOCK|
    LEFT\s+BUNDLE\s+BRANCH\s+BLOCK|
    RIGHT\s+BUNDLE\s+BRANCH\s+BLOCK|
    VENTRICULAR\s+TACHYCARDIA|
    ATRIAL\s+FIBRILLATION|
    ATRIAL\s+FLUTTER|
    VENTRICULAR\s+HYPERTROPHY|
    PROLONGED\s+QT|
    LONG\s+QTc|
    PERICARDITIS|
    ST\s+ELEVATION.*INFARCT|
    WIDE[\s\-]QRS\s+TACHYCARDIA|
    COMPLETE\s+HEART\s+BLOCK|
    EXTREME\s+TACHYCARDIA|
    VENTRICULAR\s+PREEXCITATION|
    WPW\s+PATTERN
""", re.IGNORECASE | re.VERBOSE)

neutral_patterns = re.compile(r"""
    BORDERLINE|
    POSSIBLE(?!\s+ACUTE)|
    PROBABLE(?!\s+ACUTE)|
    NONSPECIFIC|
    NON[\s\-]SPECIFIC|
    CANNOT\s+RULE\s+OUT|
    MAY\s+BE\s+DUE|
    EQUIVOCAL|
    ABNORMAL\s+FOR\s+AGE|
    CONSIDER(?!\s+ACUTE)|
    AGE\s+UNDETERMINED|
    AGE\s+INDETERMINATE|
    ST[\s\-]T\s+CHANGES|
    T\s+WAVE\s+CHANGES|
    REPOLARIZATION\s+ABNORMALITY|
    CONDUCTION\s+DEFECT|
    AXIS\s+DEVIATION|
    ATRIAL\s+ABNORMALITY|
    ATRIAL\s+ENLARGEMENT|
    LOW\s+QRS\s+VOLTAGE|
    POOR\s+R\s+WAVE\s+PROGRESSION|
    ALL\s+12\s+LEADS\s+ARE\s+MISSING|
    NOT\s+ENOUGH\s+LEADS|
    DATA\s+QUALITY|
    UNSUITABLE\s+FOR\s+ANALYSIS|
    RECORDING\s+UNSUITABLE|
    TECHNICAL\s+ERROR|
    ANALYSIS\s+ERROR|
    PEDIATRIC|
    MEASUREMENT\s+ERROR|
    NO\s+FURTHER\s+ANALYSIS
""", re.IGNORECASE | re.VERBOSE)

# Anchored to full cell value — strip() is called before matching,
# so ^ and $ reliably catch cells that are purely these phrases
normal_patterns = re.compile(r"""
    ^\s*NORMAL\s+ECG\s*$|
    WITHIN\s+NORMAL\s+LIMITS|
    NORMAL\s+VARIANT|
    AVAILABLE\s+LEADS\s+NORMAL|
    NORMAL\s+ECG\s+BASED\s+ON|
    NORMAL\s+ECG\s+EXCEPT\s+FOR\s+RATE|
    ^SINUS\s+RHYTHM\s*\.?\s*$
""", re.IGNORECASE | re.VERBOSE)

In [6]:
# Classify each cell
# Priority order: negative > neutral > normal > empty
# A cell with "POSSIBLE ACUTE INFARCT" matches both neutral (POSSIBLE) and
# negative (ACUTE INFARCT) — negative wins because it is checked first
def classify_cell(value):
    if value is None or str(value).strip() in ["nan", "", "None"]:
        return "empty"
    v = str(value).strip()
    if negative_patterns.search(v):
        return "negative"
    elif neutral_patterns.search(v):
        return "neutral"
    elif normal_patterns.search(v):
        return "normal"
    else:
        return "neutral"   # unrecognized findings treated as neutral, not normal

In [7]:
# Apply classification and ordinal encoding
report_cols = [f"report_{i}" for i in range(18)]
label_cols  = [f"report_{i}_label" for i in range(18)]
score_cols  = [f"report_{i}_label_score" for i in range(18)]

# Ordinal: empty=0, normal=1, neutral=2, negative=3
ordinal_map = {"empty": 0, "normal": 1, "neutral": 2, "negative": 3}

for report_col, label_col, score_col in zip(report_cols, label_cols, score_cols):
    machine_measurements_df[label_col] = (
        machine_measurements_df[report_col].apply(classify_cell)
    )
    machine_measurements_df[score_col] = (
        machine_measurements_df[label_col].map(ordinal_map)
    )

In [8]:
# Flag patients with no ECG conducted
# All 18 label columns empty = no ECG was performed or recorded
machine_measurements_df["no_ecg"] = (
    machine_measurements_df[label_cols]
    .eq("empty")
    .all(axis=1)
    .astype(int)
)

In [9]:
# Aggregate summary features
machine_measurements_df["ecg_negative_count"] = (
    machine_measurements_df[label_cols].eq("negative").sum(axis=1)
)
machine_measurements_df["ecg_neutral_count"] = (
    machine_measurements_df[label_cols].eq("neutral").sum(axis=1)
)
machine_measurements_df["ecg_normal_count"] = (
    machine_measurements_df[label_cols].eq("normal").sum(axis=1)
)
machine_measurements_df["ecg_max_severity"] = (
    machine_measurements_df[score_cols].max(axis=1)
)
# Mean severity excludes empty cells (score=0) so the average reflects
# only findings that were actually present
machine_measurements_df["ecg_mean_severity"] = (
    machine_measurements_df[score_cols]
    .replace(0, pd.NA)     # exclude empty (ordinal 0) from the mean
    .mean(axis=1)
    .fillna(0)             # patients with all-empty scores get 0
    .round(2)
)
machine_measurements_df["ecg_any_negative"] = (
    machine_measurements_df[label_cols].eq("negative").any(axis=1).astype(int)
)
machine_measurements_df["ecg_total_findings"] = (
    machine_measurements_df[label_cols].ne("empty").sum(axis=1)
)

/tmp/ipykernel_1020/163691471.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)             # patients with all-empty scores get 0


In [10]:
# Final table
ecg_features = [
    "subject_id",
    "ecg_any_negative",
    "ecg_max_severity",
    "ecg_negative_count",
    "ecg_neutral_count",
    "ecg_normal_count",
    "ecg_mean_severity",
    "ecg_total_findings",
    "no_ecg",
]

ecg_summary_df = machine_measurements_df[ecg_features].copy()

print(ecg_summary_df.describe())
print(f"\nPatients with no ECG conducted:              {ecg_summary_df['no_ecg'].sum():,}")
print(f"No ECG rate:                                 {ecg_summary_df['no_ecg'].mean():.1%}")
print(f"Patients with at least one negative finding: {ecg_summary_df['ecg_any_negative'].sum():,}")
print(f"Negative finding rate:                       {ecg_summary_df['ecg_any_negative'].mean():.1%}")

            subject_id  ecg_any_negative  ecg_max_severity  \
count         800035.0     800035.000000     800035.000000   
mean   15003193.384267          0.464022          2.305433   
std     2877969.929111          0.498704          0.727548   
min         10000032.0          0.000000          0.000000   
25%         12503687.0          0.000000          2.000000   
50%         15000407.0          0.000000          2.000000   
75%         17496985.0          1.000000          3.000000   
max         19999987.0          1.000000          3.000000   

       ecg_negative_count  ecg_neutral_count  ecg_normal_count  \
count       800035.000000      800035.000000     800035.000000   
mean             0.752360           2.012053          0.685583   
std              0.991838           1.391983          0.711073   
min              0.000000           0.000000          0.000000   
25%              0.000000           1.000000          0.000000   
50%              0.000000           2.000000 